In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

In [11]:
# Requires: pip install kagglehub torch torchvision
# One-time setup: create a Kaggle account -> Account Settings -> "Create New API Token"
# This downloads kaggle.json - place it at ~/.kaggle/kaggle.json (kagglehub will prompt you the first time)
import kagglehub

path = kagglehub.dataset_download("utkarshsaxenadn/car-vs-bike-classification-dataset")
print("Dataset downloaded to:", path)

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 103M/103M [00:10<00:00, 10.2MB/s] 

Extracting files...


Dataset downloaded to: /Users/keshav12/.cache/kagglehub/datasets/utkarshsaxenadn/car-vs-bike-classification-dataset/versions/1


In [12]:
# The dataset ships as nested folders; this finds the folder that directly
# contains the two class subfolders (Car / Bike) whatever the exact path looks like.
def find_class_dir(root):
    for dirpath, dirnames, filenames in os.walk(root):
        if len(dirnames) >= 2:
            return dirpath
    return root

data_dir = find_class_dir(path)
print("Using data directory:", data_dir)
print("Classes found:", os.listdir(data_dir))

Using data directory: /Users/keshav12/.cache/kagglehub/datasets/utkarshsaxenadn/car-vs-bike-classification-dataset/versions/1/Car-Bike-Dataset
Classes found: ['Car', 'Bike']


In [13]:
img_size = 150
batch_size = 20

transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor()  # also scales pixel values from 0-255 to 0-1
])

full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)
class_names = full_dataset.classes
print("Class names:", class_names)

val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Training samples: {train_size}, Validation samples: {val_size}")

Class names: ['Bike', 'Car']
Training samples: 3200, Validation samples: 800


In [14]:
class CarBikeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # AdaptiveAvgPool locks the output to a fixed 6x6 size no matter the input size,
        # so we don't have to hand-calculate the flattened dimension.
        self.adaptive_pool = nn.AdaptiveAvgPool2d((6, 6))
        self.fc1 = nn.Linear(128 * 6 * 6, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)  # raw logits - sigmoid applied inside the loss function below
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CarBikeCNN().to(device)
print(model)

CarBikeCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (adaptive_pool): AdaptiveAvgPool2d(output_size=(6, 6))
  (fc1): Linear(in_features=4608, out_features=512, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=512, out_features=1, bias=True)
)


In [15]:
criterion = nn.BCEWithLogitsLoss()  # combines Sigmoid + Binary Crossentropy in one stable step
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [16]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    print(f"Epoch {epoch+1}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f}")

/opt/homebrew/lib/python3.11/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/10 - loss: 0.5485 - accuracy: 0.7278
Epoch 2/10 - loss: 0.2716 - accuracy: 0.8875
Epoch 3/10 - loss: 0.2264 - accuracy: 0.9069
Epoch 4/10 - loss: 0.1896 - accuracy: 0.9247
Epoch 5/10 - loss: 0.1578 - accuracy: 0.9366
Epoch 6/10 - loss: 0.1430 - accuracy: 0.9437
Epoch 7/10 - loss: 0.1275 - accuracy: 0.9459
Epoch 8/10 - loss: 0.1107 - accuracy: 0.9581
Epoch 9/10 - loss: 0.0984 - accuracy: 0.9628
Epoch 10/10 - loss: 0.0967 - accuracy: 0.9650


In [18]:
model.eval()
val_loss, correct, total = 0.0, 0, 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"\nValidation Loss: {val_loss/total:.4f}")
print(f"Validation Accuracy: {correct/total:.4f}")


Validation Loss: 0.1387
Validation Accuracy: 0.9500


In [19]:
images, labels = next(iter(val_loader))
images = images.to(device)

with torch.no_grad():
    outputs = model(images)
    predicted_classes = (torch.sigmoid(outputs) > 0.5).int().flatten().cpu().numpy()

actual_classes = labels.int().numpy()

print("Predicted classes:", [class_names[c] for c in predicted_classes[:10]])
print("Actual classes:   ", [class_names[c] for c in actual_classes[:10]])

Predicted classes: ['Bike', 'Car', 'Bike', 'Car', 'Bike', 'Car', 'Bike', 'Bike', 'Car', 'Car']
Actual classes:    ['Bike', 'Car', 'Bike', 'Car', 'Bike', 'Car', 'Bike', 'Bike', 'Car', 'Car']
